# 03 — Baseline Models: Logistic Regression and Random Forest

**Owner: Member B**

This notebook compares Logistic Regression (LR) and Random Forest (RF), each with and without K-Means cluster features.

- Input: split CSVs from Notebook 01 and `output/cluster_assignments.csv` from Notebook 02.
- Target: `Repurchase_Label`.
- Feature sets: RFM only vs. RFM + cluster one-hot labels.
- Cluster features are merged by `customer_id`, never by row order.
- Hyperparameter/configuration selection uses the validation set. The test set is evaluated only once for the selected configuration.
- Main metrics: F1 and ROC-AUC; accuracy is included only as a secondary metric because the target is imbalanced.

## 1. Imports and input files

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, accuracy_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

warnings.filterwarnings("ignore")
SEED = 42
ROOT = Path(".")
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

FEATURES = ["Recency", "Frequency", "Monetary"]
TARGET = "Repurchase_Label"

def read_split(filename):
    path = ROOT / filename
    if not path.exists():
        raise FileNotFoundError(f"{filename} not found. Run Notebook 01 first.")
    df = pd.read_csv(path)
    needed = ["customer_id", *FEATURES, TARGET]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{filename} is missing columns: {missing}")
    return df

train = read_split("train_70.csv")
val = read_split("val_15.csv")
test = read_split("test_15.csv")

assignment_path = OUTPUT_DIR / "cluster_assignments.csv"
if not assignment_path.exists():
    assignment_path = ROOT / "cluster_assignments.csv"
if not assignment_path.exists():
    raise FileNotFoundError("cluster_assignments.csv not found. Run 02_clustering.ipynb first.")

assignments = pd.read_csv(assignment_path)
required_assignment_cols = {"customer_id", "cluster", "split"}
if not required_assignment_cols.issubset(assignments.columns):
    raise ValueError(f"Cluster assignments need columns {required_assignment_cols}")

print("Train/Val/Test:", train.shape, val.shape, test.shape)
print("Cluster assignments:", assignments.shape)
display(train.head())

## 2. Merge cluster labels and validate the join

In [ ]:
def merge_clusters(df, split_name):
    part = assignments.loc[assignments["split"] == split_name].copy()
    if part["customer_id"].duplicated().any():
        raise ValueError(f"Duplicate customer IDs in cluster assignments for {split_name}")
    merged = df.merge(part.drop(columns=["split"]), on="customer_id", how="left", validate="one_to_one")
    if merged["cluster"].isna().any():
        raise ValueError(f"Some {split_name} users have no cluster assignment.")
    return merged

train_m = merge_clusters(train, "train")
val_m = merge_clusters(val, "val")
test_m = merge_clusters(test, "test")

cluster_values = sorted(assignments["cluster"].dropna().astype(int).unique())
cluster_dummy_cols = [f"cluster_{k}" for k in cluster_values]

def add_cluster_onehot(df):
    out = df[FEATURES].copy()
    dummies = pd.get_dummies(df["cluster"].astype(int), prefix="cluster")
    dummies = dummies.reindex(columns=cluster_dummy_cols, fill_value=0).astype(int)
    return pd.concat([out.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)

y_train = train_m[TARGET].astype(int).to_numpy()
y_val = val_m[TARGET].astype(int).to_numpy()
y_test = test_m[TARGET].astype(int).to_numpy()

print("Cluster categories:", cluster_values)
print("Train label distribution:", pd.Series(y_train).value_counts().to_dict())
print("Validation positive rate:", round(y_val.mean(), 4))
print("Test positive rate:", round(y_test.mean(), 4))

## 3. Prepare model-specific feature matrices

In [ ]:
# LR benefits from standardized numeric features. The scaler is fit on training data only.
imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

Xtr_rfm_imp = imputer.fit_transform(train_m[FEATURES])
Xva_rfm_imp = imputer.transform(val_m[FEATURES])
Xte_rfm_imp = imputer.transform(test_m[FEATURES])

Xtr_rfm_scaled = scaler.fit_transform(Xtr_rfm_imp)
Xva_rfm_scaled = scaler.transform(Xva_rfm_imp)
Xte_rfm_scaled = scaler.transform(Xte_rfm_imp)

# Build the same feature-set comparison for both algorithms.
def with_onehot(scaled_rfm, df):
    dummies = pd.get_dummies(df["cluster"].astype(int), prefix="cluster")
    dummies = dummies.reindex(columns=cluster_dummy_cols, fill_value=0).to_numpy(dtype=float)
    return np.concatenate([scaled_rfm, dummies], axis=1)

X = {
    ("Logistic Regression", "RFM"): (Xtr_rfm_scaled, Xva_rfm_scaled, Xte_rfm_scaled),
    ("Logistic Regression", "RFM + Cluster One-hot"): (
        with_onehot(Xtr_rfm_scaled, train_m),
        with_onehot(Xva_rfm_scaled, val_m),
        with_onehot(Xte_rfm_scaled, test_m)
    ),
    ("Random Forest", "RFM"): (
        Xtr_rfm_imp, Xva_rfm_imp, Xte_rfm_imp
    ),
    ("Random Forest", "RFM + Cluster One-hot"): (
        np.concatenate([Xtr_rfm_imp, train_m[cluster_dummy_cols_from_frame(train_m)].to_numpy()], axis=1)
        if False else np.concatenate([Xtr_rfm_imp, pd.get_dummies(train_m["cluster"].astype(int)).reindex(columns=cluster_values, fill_value=0).to_numpy()], axis=1),
        np.concatenate([Xva_rfm_imp, pd.get_dummies(val_m["cluster"].astype(int)).reindex(columns=cluster_values, fill_value=0).to_numpy()], axis=1),
        np.concatenate([Xte_rfm_imp, pd.get_dummies(test_m["cluster"].astype(int)).reindex(columns=cluster_values, fill_value=0).to_numpy()], axis=1)
    )
}

def safe_auc(y, prob):
    return float(roc_auc_score(y, prob)) if len(np.unique(y)) == 2 else float("nan")

def metrics(y, prob, threshold=0.5):
    pred = (prob >= threshold).astype(int)
    return {
        "F1": f1_score(y, pred, zero_division=0),
        "AUC": safe_auc(y, prob),
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(y, pred, zero_division=0),
        "Recall": recall_score(y, pred, zero_division=0)
    }

def make_model(name):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED)
    return RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=SEED,
        n_jobs=-1, min_samples_leaf=2
    )

print("Feature matrix dimensions:")
for key, matrices in X.items():
    print(key, matrices[0].shape, matrices[1].shape, matrices[2].shape)

## 4. Train and compare configurations on the validation set

In [ ]:
validation_rows = []
fitted_models = {}
validation_predictions = {}

for (model_name, feature_name), (Xtr, Xva, Xte) in X.items():
    model = make_model(model_name)
    model.fit(Xtr, y_train)
    val_prob = model.predict_proba(Xva)[:, 1]
    row = {
        "Model": model_name,
        "Feature Set": feature_name,
        **{f"Val {k}": v for k, v in metrics(y_val, val_prob).items()}
    }
    validation_rows.append(row)
    key = (model_name, feature_name)
    fitted_models[key] = model
    validation_predictions[key] = val_prob

val_results = pd.DataFrame(validation_rows).sort_values(
    ["Val F1", "Val AUC"], ascending=False, na_position="last"
).reset_index(drop=True)
val_results.to_csv(OUTPUT_DIR / "baseline_validation_results.csv", index=False)
display(val_results)

plt.figure(figsize=(8,4))
plot_df = val_results.melt(
    id_vars=["Model","Feature Set"], value_vars=["Val F1","Val AUC"],
    var_name="Metric", value_name="Score"
)
plot_df["Configuration"] = plot_df["Model"] + "\n" + plot_df["Feature Set"]
sns.barplot(data=plot_df, x="Configuration", y="Score", hue="Metric")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.title("Baseline model comparison (validation set)")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_validation_comparison.png", dpi=160)
plt.show()
plt.close()

## 5. Select by validation F1 and evaluate the test set once

In [ ]:
# Select one configuration using validation metrics, then evaluate that configuration on test.
best_row = val_results.iloc[0]
best_key = (best_row["Model"], best_row["Feature Set"])
best_model = fitted_models[best_key]
best_X_test = X[best_key][2]

test_prob = best_model.predict_proba(best_X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)
test_metrics = metrics(y_test, test_prob)

test_result = {
    "Model": best_key[0],
    "Feature Set": best_key[1],
    **{f"Test {k}": v for k, v in test_metrics.items()}
}
print("Selected configuration based on validation F1:", best_key)
display(pd.DataFrame([test_result]).round(4))
pd.DataFrame([test_result]).to_csv(OUTPUT_DIR / "baseline_selected_test_result.csv", index=False)

print(classification_report(y_test, test_pred, zero_division=0))
cm = confusion_matrix(y_test, test_pred, labels=[0,1])
disp = ConfusionMatrixDisplay(cm, display_labels=["No repurchase (0)", "Repurchase (1)"])
disp.plot(values_format="d")
plt.title(f"Selected baseline — test confusion matrix\n{best_key[0]} / {best_key[1]}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "baseline_selected_test_confusion_matrix.png", dpi=160)
plt.show()
plt.close()

# Save selected estimator and the exact preprocessing objects for reproducibility.
joblib.dump({
    "model": best_model,
    "model_name": best_key[0],
    "feature_set": best_key[1],
    "imputer": imputer,
    "scaler": scaler,
    "feature_columns": FEATURES,
    "cluster_dummy_columns": cluster_dummy_cols,
    "seed": SEED
}, OUTPUT_DIR / "selected_baseline_model.joblib")

## 6. Export a handoff summary for the final report

`baseline_validation_results.csv` contains all four validation configurations. `baseline_selected_test_result.csv` contains test metrics for the configuration selected on validation data.

Do not compare the selected model's test score with several other test scores and then pick the best test result; that would leak test information into model selection.

In [ ]:
# Compact validation table is the main comparison table for reporting.
report_table = val_results[["Model","Feature Set","Val F1","Val AUC","Val Accuracy","Val Precision","Val Recall"]].copy()
display(report_table.round(4))
print("Saved outputs:")
for p in sorted(OUTPUT_DIR.glob("*")):
    print(" -", p.name)